In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import keras

# Data loading, model definition and labels are shared with train.py / realtimedetection.py
from train import load_dataset, build_model, LABELS, TRAIN_DIR, TEST_DIR, MODEL_PATH
%matplotlib inline

In [ ]:
x_train, y_train = load_dataset(TRAIN_DIR)
x_test, y_test = load_dataset(TEST_DIR)
print(x_train.shape, x_test.shape)

In [ ]:
counts = np.bincount(y_train, minlength=len(LABELS))
weights = np.sqrt(counts.sum() / (len(LABELS) * counts))
class_weight = {i: float(w) for i, w in enumerate(weights)}

y_train_oh = keras.utils.to_categorical(y_train, len(LABELS))
y_test_oh = keras.utils.to_categorical(y_test, len(LABELS))

In [ ]:
model = build_model()
model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss=keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=['accuracy'],
)
model.summary()

In [ ]:
callbacks = [
    keras.callbacks.ModelCheckpoint(MODEL_PATH, monitor='val_accuracy', save_best_only=True, verbose=1),
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, min_lr=1e-5, verbose=1),
    keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=12, restore_best_weights=True, verbose=1),
]
history = model.fit(x_train, y_train_oh, batch_size=128, epochs=60,
                    validation_data=(x_test, y_test_oh),
                    class_weight=class_weight, callbacks=callbacks)

In [ ]:
plt.plot(history.history['accuracy'], label='train')
plt.plot(history.history['val_accuracy'], label='test')
plt.xlabel('epoch'); plt.ylabel('accuracy'); plt.legend(); plt.show()

In [ ]:
# Best checkpoint is saved by ModelCheckpoint; reload it to verify
model = keras.models.load_model(MODEL_PATH)
loss, acc = model.evaluate(x_test, y_test_oh, verbose=0)
print(f"Test accuracy: {acc:.4f}")

In [ ]:
def predict_sample(emotion, split='test'):
    folder = os.path.join('images', split, emotion)
    name = next(f for f in sorted(os.listdir(folder)) if f.lower().endswith(('.jpg', '.jpeg', '.png')))
    img = keras.utils.load_img(os.path.join(folder, name), color_mode='grayscale', target_size=(48, 48))
    feature = np.array(img, dtype=np.float32).reshape(1, 48, 48, 1) / 255.0
    pred = LABELS[model.predict(feature, verbose=0).argmax()]
    plt.imshow(feature.reshape(48, 48), cmap='gray')
    plt.title(f"Original: {emotion} | Predicted: {pred}")
    plt.axis('off'); plt.show()

for emotion in LABELS:
    predict_sample(emotion)